In [ ]:
!pip install transformers accelerate peft bitsandbytes pillow torch torchvision qwen-vl-utils
!pip install alive-progress

   ---------------------------------------- 0.0/556.4 kB ? eta -:--:--
   ---------------------------------------- 556.4/556.4 kB 3.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/59.0 MB ? eta -:--:--
   ------ --------------------------------- 10.0/59.0 MB 47.5 MB/s eta 0:00:02
   -------------- ------------------------- 20.7/59.0 MB 48.5 MB/s eta 0:00:01
   -------------------- ------------------- 30.7/59.0 MB 48.7 MB/s eta 0:00:01
   -------------------------- ------------- 39.6/59.0 MB 46.6 MB/s eta 0:00:01
   --------------------------------- ------ 49.8/59.0 MB 47.3 MB/s eta 0:00:01
   ---------------------------------------  59.0/59.0 MB 47.0 MB/s eta 0:00:01
   ---------------------------------------- 59.0/59.0 MB 43.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/32.3 MB ? eta -:--:--
   ------- -------------------------------- 6.3/32.3 MB 29.7 MB/s eta 0:00:01
   --------------- ------------------------ 12.6/32.3 MB 30.3 MB/s eta 0:00:01

In [2]:
import json
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from transformers import TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sklearn.preprocessing import StandardScaler
import pickle
from alive_progress import alive_bar
from qwen_vl_utils import process_vision_info

c:\Users\rohai\anaconda3\envs\prac\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Data normalization class (same as original)
class EngagementNormalizer:
    """Normalize engagement metrics for better training"""
    def __init__(self):
        self.scaler = StandardScaler()
        
    def fit(self, engagement_data):
        """Fit on training data"""
        engagement_array = np.array([[
            e['likes'],
            e['comments'],
            e['views']
        ] for e in engagement_data])
        self.scaler.fit(engagement_array)
        
    def transform(self, engagement):
        """Transform engagement dict to normalized array"""
        arr = np.array([[
            engagement['likes'],
            engagement['comments'],
            engagement['views']
        ]])
        return self.scaler.transform(arr)[0]
    
    def inverse_transform(self, normalized_engagement):
        """Convert normalized predictions back to original scale"""
        return self.scaler.inverse_transform(normalized_engagement)

In [4]:
class QwenEngagementDataset(Dataset):
    """Dataset for Qwen2.5-VL model with engagement prediction"""
    def __init__(self, data, processor, normalizer):
        self.data = data
        self.processor = processor
        self.normalizer = normalizer
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Load image
        image = Image.open(item['image']).convert('RGB')
        
        # Create text prompt
        text_prompt = f"""Predict the engagement metrics for this content.
Title: {item['title']}
Tag: {item['tag']}
Description: {item['description']}

Provide the predicted likes, comments, and views as three numbers separated by commas."""
        
        # Normalize engagement metrics
        normalized_engagement = self.normalizer.transform(item['engagement'])
        
        # Format target as string for model output
        target_text = f"{normalized_engagement[0]:.4f},{normalized_engagement[1]:.4f},{normalized_engagement[2]:.4f}"
        
        # Prepare messages format for Qwen2.5-VL
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": text_prompt}
                ]
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": target_text}]
            }
        ]
        
        # Process with Qwen processor
        text = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        
        image_inputs, video_inputs = process_vision_info(messages)
        
        inputs = self.processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        )
        
        # Remove batch dimension
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        
        # Add labels for training
        inputs['labels'] = inputs['input_ids'].clone()
        
        return inputs

In [ ]:
def setup_qwen_model(model_name="Qwen/Qwen2.5-VL-7B-Instruct", use_4bit=True):
    # Load processor
    processor = AutoProcessor.from_pretrained(model_name)
    
    # Configure 4-bit quantization if requested
    if use_4bit:
        from transformers import BitsAndBytesConfig
        
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )
        
        model = Qwen2VLForConditionalGeneration.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=torch.bfloat16
        )
    else:
        model = Qwen2VLForConditionalGeneration.from_pretrained(
            model_name,
            device_map="auto",
            torch_dtype=torch.bfloat16
        )
    
    # Prepare model for training
    model = prepare_model_for_kbit_training(model)
    
    # Configure LoRA
    lora_config = LoraConfig(
        r=64,  # LoRA rank
        lora_alpha=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )
    
    # Apply LoRA
    model = get_peft_model(model, lora_config)
    
    print(f"Trainable parameters: {model.print_trainable_parameters()}")
    
    return model, processor

In [ ]:
def train_qwen_engagement_model(
    train_path,
    val_path,
    output_dir="./qwen_engagement_model",
    batch_size=4,
    num_epochs=10,
    learning_rate=2e-4,
    use_4bit=True
):
    
    # Load data
    with open(train_path, 'r') as f:
        train_data = json.load(f)
    
    with open(val_path, 'r') as f:
        val_data = json.load(f)
    
    print(f"Loaded {len(train_data)} training samples and {len(val_data)} validation samples")
    
    # Initialize normalizer
    normalizer = EngagementNormalizer()
    normalizer.fit([item['engagement'] for item in train_data])
    
    # Save normalizer
    with open('qwen_engagement_normalizer.pkl', 'wb') as f:
        pickle.dump(normalizer, f)
    
    # Setup model and processor
    print("Loading Qwen2.5-VL model...")
    model, processor = setup_qwen_model(use_4bit=use_4bit)
    
    # Create datasets
    train_dataset = QwenEngagementDataset(train_data, processor, normalizer)
    val_dataset = QwenEngagementDataset(val_data, processor, normalizer)
    
    # Training arguments
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=4,
        learning_rate=learning_rate,
        warmup_steps=100,
        logging_steps=10,
        eval_steps=50,
        save_steps=100,
        evaluation_strategy="steps",
        save_strategy="steps",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        fp16=False,
        bf16=True,
        optim="paged_adamw_8bit",
        max_grad_norm=0.3,
        report_to="none"
    )
    
    # Custom data collator
    def collate_fn(batch):
        # Find max length in batch
        max_length = max([item['input_ids'].shape[0] for item in batch])
        
        # Pad all sequences
        input_ids = []
        attention_mask = []
        labels = []
        pixel_values = []
        image_grid_thw = []
        
        for item in batch:
            seq_len = item['input_ids'].shape[0]
            pad_len = max_length - seq_len
            
            # Pad input_ids and attention_mask
            input_ids.append(torch.cat([
                item['input_ids'],
                torch.full((pad_len,), processor.tokenizer.pad_token_id)
            ]))
            
            attention_mask.append(torch.cat([
                item['attention_mask'],
                torch.zeros(pad_len)
            ]))
            
            labels.append(torch.cat([
                item['labels'],
                torch.full((pad_len,), -100)
            ]))
            
            pixel_values.append(item['pixel_values'])
            image_grid_thw.append(item['image_grid_thw'])
        
        return {
            'input_ids': torch.stack(input_ids),
            'attention_mask': torch.stack(attention_mask),
            'labels': torch.stack(labels),
            'pixel_values': torch.cat(pixel_values, dim=0),
            'image_grid_thw': torch.cat(image_grid_thw, dim=0)
        }
    
    # Initialize trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=collate_fn
    )
    
    # Train
    print("Starting training...")
    trainer.train()
    
    # Save final model
    print("Saving final model...")
    trainer.save_model(f"{output_dir}/final_model")
    processor.save_pretrained(f"{output_dir}/final_model")
    
    return model, processor, normalizer

In [ ]:
def predict_engagement_qwen(
    model_path,
    image_path,
    title,
    tag,
    description,
    normalizer_path="qwen_engagement_normalizer.pkl"
):
    
    # Load normalizer
    with open(normalizer_path, 'rb') as f:
        normalizer = pickle.load(f)
    
    # Load model and processor
    processor = AutoProcessor.from_pretrained(model_path)
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        model_path,
        device_map="auto",
        torch_dtype=torch.bfloat16
    )
    model.eval()
    
    # Load and prepare image
    image = Image.open(image_path).convert('RGB')
    
    # Create prompt
    text_prompt = f"""Predict the engagement metrics for this content.
Title: {title}
Tag: {tag}
Description: {description}

Provide the predicted likes, comments, and views as three numbers separated by commas."""
    
    # Prepare messages
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": text_prompt}
            ]
        }
    ]
    
    # Process inputs
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    image_inputs, video_inputs = process_vision_info(messages)
    
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to(model.device)
    
    # Generate prediction
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=False
        )
    
    # Decode output
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    
    output_text = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]
    
    # Parse output
    try:
        normalized_values = [float(x.strip()) for x in output_text.split(',')]
        normalized_array = np.array([normalized_values])
        
        # Denormalize
        denormalized = normalizer.inverse_transform(normalized_array)[0]
        
        engagement = {
            'likes': int(max(0, denormalized[0])),
            'comments': int(max(0, denormalized[1])),
            'views': int(max(0, denormalized[2]))
        }
    except:
        print(f"Could not parse output: {output_text}")
        engagement = {'likes': 0, 'comments': 0, 'views': 0}
    
    return engagement, output_text

In [ ]:
# Train the model
model, processor, normalizer = train_qwen_engagement_model(
    train_path='../../data/baby_train.json', #TODO: adjust path, batch size and epochs
    val_path='../../data/baby_test.json',
    output_dir='./qwen_engagement_model',
    batch_size=2,  
    num_epochs=1,
    learning_rate=2e-4,
    use_4bit=True
)

In [ ]:

engagement, raw_output = predict_engagement_qwen(
    model_path='./qwen_engagement_model/final_model',
    image_path='path/to/test/image.jpg',
    title='Test Title',
    tag='Test Tag',
    description='Test Description'
)

print("Predicted Engagement:", engagement)
print("Raw Model Output:", raw_output)